In [3]:
import pandas as pd
import numpy as np
import torch
import glob
import os
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("--- BƯỚC 1: NẠP DATA & CHUYỂN ĐỔI KIỂU DỮ LIỆU ---")

# 1. Quét danh sách file
data_pattern = '/kaggle/input/datasets/huy291/criteo-cleaned-data/data/*.parquet'
all_files = sorted(glob.glob(data_pattern))
print(f"[*] Tìm thấy {len(all_files)} file parquet.")

# 2. Nạp số lượng file mong muốn (Ví dụ: 5 file)
num_files_to_load = 50 
files_to_process = all_files[:num_files_to_load]

list_df = []
for f in files_to_process:
    list_df.append(pd.read_parquet(f))
    print(f"   + Đã nạp: {os.path.basename(f)}")

df = pd.concat(list_df, ignore_index=True)
print(f"[*] Tổng số mẫu: {len(df):,}")

# 3. Định nghĩa các cột
dense_cols = [f'I{i}' for i in range(1, 14)]
sparse_cols = [f'C{i}' for i in range(1, 27)]
target_col = 'label'

# 4. CHUYỂN ĐỔI CHUỖI SANG SỐ NGUYÊN (Sửa lỗi TypeError tại đây)
print("[*] Đang mã hóa Sparse Features từ Chuỗi sang Số nguyên...")
vocab_sizes = []
for col in sparse_cols:
    le = LabelEncoder()
    # Ép kiểu sang string để xử lý đồng nhất cả "unknown" và các giá trị khác
    df[col] = le.fit_transform(df[col].astype(str))
    vocab_sizes.append(len(le.classes_))

# 5. Ép kiểu cho Dense Features và Label (Đảm bảo an toàn cho Tensor)
for col in dense_cols:
    df[col] = df[col].astype(np.float32)
df[target_col] = df[target_col].astype(np.float32)

# 6. Chia dữ liệu Train/Validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 7. Tạo DataLoader
def create_dataloader(data_df, batch_size=2048):
    X_dense = torch.tensor(data_df[dense_cols].values, dtype=torch.float32)
    X_sparse = torch.tensor(data_df[sparse_cols].values, dtype=torch.long)
    y = torch.tensor(data_df[target_col].values, dtype=torch.float32)
    return DataLoader(TensorDataset(X_dense, X_sparse, y), batch_size=batch_size, shuffle=True)

train_loader = create_dataloader(train_df)
val_loader = create_dataloader(val_df)

print(f"\n[*] Hoàn tất chuẩn bị: Train ({len(train_df):,} mẫu), Val ({len(val_df):,} mẫu)")

--- BƯỚC 1: NẠP DATA & CHUYỂN ĐỔI KIỂU DỮ LIỆU ---
[*] Tìm thấy 50 file parquet.
   + Đã nạp: train-00000-of-00050.parquet
   + Đã nạp: train-00001-of-00050.parquet
   + Đã nạp: train-00002-of-00050.parquet
   + Đã nạp: train-00003-of-00050.parquet
   + Đã nạp: train-00004-of-00050.parquet
   + Đã nạp: train-00005-of-00050.parquet
   + Đã nạp: train-00006-of-00050.parquet
   + Đã nạp: train-00007-of-00050.parquet
   + Đã nạp: train-00008-of-00050.parquet
   + Đã nạp: train-00009-of-00050.parquet
   + Đã nạp: train-00010-of-00050.parquet
   + Đã nạp: train-00011-of-00050.parquet
   + Đã nạp: train-00012-of-00050.parquet
   + Đã nạp: train-00013-of-00050.parquet
   + Đã nạp: train-00014-of-00050.parquet
   + Đã nạp: train-00015-of-00050.parquet
   + Đã nạp: train-00016-of-00050.parquet
   + Đã nạp: train-00017-of-00050.parquet
   + Đã nạp: train-00018-of-00050.parquet
   + Đã nạp: train-00019-of-00050.parquet
   + Đã nạp: train-00020-of-00050.parquet
   + Đã nạp: train-00021-of-00050.par

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

print("--- BƯỚC 2: KHỞI TẠO KIẾN TRÚC MÔ HÌNH ---")

class FeatureRoutedAttentionMoE(nn.Module):
    def __init__(self, num_dense, vocab_sizes, emb_dim=32, k=10, num_heads=4):
        super().__init__()
        self.k = k
        self.num_total_features = num_dense + len(vocab_sizes)
        
        # Nhúng Dense và Sparse
        self.dense_embs = nn.ModuleList([nn.Linear(1, emb_dim) for _ in range(num_dense)])
        self.sparse_embs = nn.ModuleList([nn.Embedding(v, emb_dim) for v in vocab_sizes])
        
        # Field Embedding (Giữ vị trí ngữ nghĩa)
        self.field_emb = nn.Embedding(self.num_total_features, emb_dim)
        
        # Feature Router
        self.router = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, self.num_total_features)
        )
        
        # Attention Expert
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=num_heads, batch_first=True, dim_feedforward=emb_dim*2)
        self.attention_expert = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.predictor = nn.Linear(emb_dim, 1)

    def forward(self, dense_x, sparse_x):
        batch_size = dense_x.size(0)
        feature_embs = []
        
        for i in range(dense_x.size(1)):
            feature_embs.append(self.dense_embs[i](dense_x[:, i].unsqueeze(1)).unsqueeze(1))
        for i in range(sparse_x.size(1)):
            feature_embs.append(self.sparse_embs[i](sparse_x[:, i]).unsqueeze(1))
            
        all_embs = torch.cat(feature_embs, dim=1)
        
        field_indices = torch.arange(self.num_total_features, device=dense_x.device).unsqueeze(0).expand(batch_size, -1)
        all_embs = all_embs + self.field_emb(field_indices)
        
        context = all_embs.mean(dim=1)
        router_probs = torch.softmax(self.router(context), dim=-1)
        topk_probs, topk_indices = torch.topk(router_probs, self.k, dim=-1)
        
        expanded_indices = topk_indices.unsqueeze(-1).expand(-1, -1, all_embs.size(-1))
        selected_embs = torch.gather(all_embs, 1, expanded_indices) * topk_probs.unsqueeze(-1)
        
        expert_out = self.attention_expert(selected_embs)
        logits = self.predictor(expert_out.mean(dim=1)).squeeze()
        return logits, topk_indices

# Khởi tạo mô hình trên thiết bị tốt nhất
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Lưu ý: vocab_sizes lấy từ Block 1
model = FeatureRoutedAttentionMoE(num_dense=13, vocab_sizes=vocab_sizes, emb_dim=32, k=10).to(device)
print(f"[*] Khởi tạo thành công mô hình trên: {device}")

--- BƯỚC 2: KHỞI TẠO KIẾN TRÚC MÔ HÌNH ---
[*] Khởi tạo thành công mô hình trên: cuda


In [6]:
from sklearn.metrics import roc_auc_score

print("--- BƯỚC 3: HUẤN LUYỆN, ĐÁNH GIÁ & PHÂN TÍCH ĐA DẠNG ---")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

def analyze_diversity(model, loader):
    model.eval()
    all_combinations = []
    print("[*] Đang phân tích các quyết định của Router trên tập Validation...")
    
    with torch.no_grad():
        for dense_batch, sparse_batch, _ in loader:
            _, topk_indices = model(dense_batch.to(device), sparse_batch.to(device))
            # Chuyển mỗi hàng (set of indices) thành một tuple đã sắp xếp để đếm tổ hợp duy nhất
            for row in topk_indices.cpu().numpy():
                all_combinations.append(tuple(sorted(row)))
    
    counts = Counter(all_combinations)
    print(f"\n[KẾT QUẢ DYNAMIC ANALYSIS]")
    print(f"- Tổng số mẫu kiểm tra: {len(all_combinations)}")
    print(f"- Số lượng tổ hợp feature khác nhau được chọn: {len(counts)}")
    print(f"- Top 3 tổ hợp phổ biến nhất (Feature Indices):")
    for i, (comb, freq) in enumerate(counts.most_common(3)):
        print(f"  #{i+1}: {comb} (Xuất hiện {freq} lần)")

# Vòng lặp Training rút gọn
for epoch in range(3):
    model.train()
    for db, sb, yb in train_loader:
        optimizer.zero_grad()
        logits, _ = model(db.to(device), sb.to(device))
        loss = criterion(logits, yb.to(device))
        loss.backward()
        optimizer.step()
    
    # Đánh giá AUC nhanh
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for db, sb, yb in val_loader:
            out, _ = model(db.to(device), sb.to(device))
            preds.extend(torch.sigmoid(out).cpu().numpy())
            labels.extend(yb.numpy())
    print(f"Epoch {epoch+1} | Val AUC: {roc_auc_score(labels, preds):.4f}")

# Chạy phân tích sau khi huấn luyện xong
analyze_diversity(model, val_loader)

--- BƯỚC 3: HUẤN LUYỆN, ĐÁNH GIÁ & PHÂN TÍCH ĐA DẠNG ---
Epoch 1 | Val AUC: 0.7812
Epoch 2 | Val AUC: 0.7877
Epoch 3 | Val AUC: 0.7858
[*] Đang phân tích các quyết định của Router trên tập Validation...


NameError: name 'Counter' is not defined

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from collections import Counter  # <-- Đây là dòng đã sửa lỗi

print("--- BƯỚC 3: HUẤN LUYỆN, ĐÁNH GIÁ & PHÂN TÍCH ĐA DẠNG ---")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

def analyze_diversity(model, loader):
    model.eval()
    all_combinations = []
    print("[*] Đang phân tích các quyết định của Router trên tập Validation...")
    
    with torch.no_grad():
        for dense_batch, sparse_batch, _ in loader:
            _, topk_indices = model(dense_batch.to(device), sparse_batch.to(device))
            # Chuyển mỗi hàng (set of indices) thành một tuple đã sắp xếp để đếm tổ hợp
            for row in topk_indices.cpu().numpy():
                all_combinations.append(tuple(sorted(row)))
    
    counts = Counter(all_combinations)
    print(f"\n[KẾT QUẢ DYNAMIC ANALYSIS]")
    print(f"- Tổng số mẫu kiểm tra: {len(all_combinations):,}")
    print(f"- Số lượng tổ hợp feature khác nhau được chọn: {len(counts):,}")
    print(f"- Top 3 tổ hợp phổ biến nhất (Feature Indices):")
    for i, (comb, freq) in enumerate(counts.most_common(3)):
        print(f"  #{i+1}: {comb} (Xuất hiện {freq:,} lần)")


# --- Phân tích Đa dạng Định tuyến ---
analyze_diversity(model, val_loader)

--- BƯỚC 3: HUẤN LUYỆN, ĐÁNH GIÁ & PHÂN TÍCH ĐA DẠNG ---
[*] Đang phân tích các quyết định của Router trên tập Validation...

[KẾT QUẢ DYNAMIC ANALYSIS]
- Tổng số mẫu kiểm tra: 9,168,124
- Số lượng tổ hợp feature khác nhau được chọn: 1,359
- Top 3 tổ hợp phổ biến nhất (Feature Indices):
  #1: (np.int64(2), np.int64(6), np.int64(7), np.int64(12), np.int64(15), np.int64(23), np.int64(27), np.int64(32), np.int64(33), np.int64(38)) (Xuất hiện 2,003,111 lần)
  #2: (np.int64(2), np.int64(6), np.int64(7), np.int64(12), np.int64(15), np.int64(17), np.int64(23), np.int64(27), np.int64(33), np.int64(38)) (Xuất hiện 1,641,252 lần)
  #3: (np.int64(2), np.int64(6), np.int64(7), np.int64(12), np.int64(15), np.int64(23), np.int64(27), np.int64(33), np.int64(35), np.int64(38)) (Xuất hiện 1,585,826 lần)
